## Przykład klasyfikacji cyfr MNIST w PyTorch i śledzenia eksperymentów w MLflow
Notebook pokazuje prosty proces machine learning dla klasyfikacji obrazów cyfr MNIST.
1. Import danych
2. Analiza danych
3. Przygotowanie danych wejściowych
4. Trening modelu
5. Ewaluacja modelu
6. Logowanie wyników w MLflow

In [ ]:
! pip install mlflow

In [ ]:
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import mlflow

# uruchom `mlflow ui` w terminalu, aby zobaczyć wyniki eksperymentów w przeglądarce

# ustaw nazwę eksperymentu
mlflow.set_tracking_uri("file:../model/mlruns")
mlflow.set_experiment("image_digit_classification_mnist")

input_size = 28*28
hidden_size = 100
output_size = 10
learning_rate = 0.001
batch_size = 100
epochs = 5

mlflow.log_param("input_size", input_size)
mlflow.log_param("hidden_size", hidden_size)
mlflow.log_param("output_size", output_size)
mlflow.log_param("learning_rate", learning_rate)
mlflow.log_param("batch_size", batch_size)
mlflow.log_param("epochs", epochs)

train_dataset = torchvision.datasets.MNIST(root='../data', train=True, transform=transforms.ToTensor(), download=True)
test_dataset = torchvision.datasets.MNIST(root='../data', train=False, transform=transforms.ToTensor())

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=batch_size, shuffle=True)

class NeurlNet(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super(NeurlNet, self).__init__()
        self.l1 = nn.Linear(input_size, hidden_size)
        self.relu = nn.ReLU()
        self.l2 = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        out = self.l1(x)
        out = self.relu(out)
        out = self.l2(out)
        return out

model = NeurlNet(input_size, hidden_size, output_size)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

for epoch in range(epochs):
    for batch_idx, (data, target) in enumerate(train_loader):
        optimizer.zero_grad()
        output = model(data.reshape(-1, input_size))
        loss = criterion(output, target)
        loss.backward()
        optimizer.step()
        mlflow.log_metric("train_loss", loss.item(), batch_idx * len(data))
        if batch_idx % 10 == 0:
            print(loss.item())

In [ ]:
plt.imshow(train_dataset[28][0].reshape(28, 28), cmap='gray')
plt.show()
plt.savefig("number_plot.png")
mlflow.log_artifact("number_plot.png")

print(train_dataset[28][1])


In [ ]:
run = mlflow.active_run()
runId = run.info.run_id

# Wytrenuj model i pobierz obiekt wytrenowanego modelu
# Zaloguj model do MLflow
mlflow.pytorch.log_model(
    pytorch_model=model,
    artifact_path=f"../model/my-model{runId}",
    registered_model_name="my-registered-model"
)
# Save the model to a file
mlflow.pytorch.save_model(
    pytorch_model=model,
    path=f"../model/my-model{runId}",
)

In [ ]:
all = 0
correct = 0
for i, (inputs, labels) in enumerate(test_loader):
    predicted = model(inputs.reshape(-1, input_size))
    all += labels.size(0)
    for j, label in enumerate(labels):
        if label.item() == predicted[j].argmax().item():
            correct += 1

print(f'Accuracy: {correct / all}')
mlflow.log_param("test_accuracy", correct / all)
mlflow.end_run()

for i, (inputs, labels) in enumerate(test_loader):
    predicted = model(inputs.reshape(-1, input_size))
    for j, label in enumerate(labels):
        print(label.item(), predicted[j].argmax().item())
        plt.imshow(inputs[j][0], cmap='gray')
        plt.show()
        if j == 10:
            break
    if i == 5:
        break